In [1]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

# load the tables needed for stock, demand and supplier waiting time
so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')
po_h = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')
products = pd.read_csv(BASE_PATH + 'products.csv')
suppliers = pd.read_csv(BASE_PATH + 'suppliers.csv')
inv = pd.read_csv(BASE_PATH + 'inventory_master.csv')
ledger = pd.read_csv(BASE_PATH + 'stock_ledger.csv')

# rows and columns of each table
tables = {'so_h': so_h, 'so_l': so_l, 'po_h': po_h, 'po_l': po_l,
          'products': products, 'suppliers': suppliers, 'inv': inv, 'ledger': ledger}
for name, t in tables.items():
    print(name, t.shape)

# column names of the tables we have not listed in full yet
for name in ['products', 'po_h', 'suppliers', 'inv']:
    print()
    print(name, 'columns:')
    print(list(tables[name].columns))

Mounted at /content/drive
so_h (20000, 11)
so_l (130402, 9)
po_h (24000, 10)
po_l (155495, 9)
products (30, 23)
suppliers (8, 12)
inv (180, 8)
ledger (237230, 9)

products columns:
['product_id', 'product_name', 'category', 'machine_type', 'brand', 'model_compatibility', 'unit_cost', 'unit_price', 'margin_percentage', 'gst_rate', 'weight_kg', 'dimensions_cm', 'material_type', 'warranty_months', 'reorder_level', 'safety_stock', 'max_stock_level', 'lead_time_days', 'criticality_level', 'usage_frequency', 'uom', 'last_purchase_price', 'last_purchase_date']

po_h columns:
['po_id', 'supplier_id', 'branch_id', 'order_date', 'expected_delivery_date', 'received_date', 'po_status', 'total_cost', 'total_gst_amount', 'grand_total']

suppliers columns:
['supplier_id', 'supplier_name', 'supplier_type', 'product_category', 'city', 'province', 'region', 'pincode', 'lead_time_days', 'reliability_score', 'import_duty_rate', 'china_tax_id']

inv columns:
['product_id', 'branch_id', 'opening_stock', 're

In [2]:
# purchase lines of Received orders with their supplier and branch
received = po_h[po_h['po_status'] == 'Received']
pl = po_l.merge(received[['po_id', 'supplier_id', 'branch_id']], on='po_id', how='inner')
print('Received purchase lines', len(pl), '(expected 140077)')
print('Received units', pl['quantity'].sum(), '(expected 22387123)')

# how many different suppliers each product comes from
per_product = pl.groupby('product_id')['supplier_id'].nunique()
print()
print('products by number of suppliers:')
print(per_product.value_counts().sort_index())

# how many different suppliers each product and branch pair comes from
per_pair = pl.groupby(['product_id', 'branch_id'])['supplier_id'].nunique()
print()
print('pairs counted', len(per_pair))
print('pairs by number of suppliers:')
print(per_pair.value_counts().sort_index())


Received purchase lines 140077 (expected 140077)
Received units 22387123 (expected 22387123)

products by number of suppliers:
supplier_id
8    30
Name: count, dtype: int64

pairs counted 180
pairs by number of suppliers:
supplier_id
8    180
Name: count, dtype: int64


In [3]:
# real waiting time of each Received order: received date minus order date, in days
received = po_h[po_h['po_status'] == 'Received'].copy()
received['lead_days'] = (pd.to_datetime(received['received_date']) - pd.to_datetime(received['order_date'])).dt.days

# attach the waiting time to every Received purchase line
pl = po_l.merge(received[['po_id', 'supplier_id', 'branch_id', 'lead_days']], on='po_id', how='inner')
print('Received purchase lines', len(pl), '(expected 140077)')
print('waiting days over all lines: smallest', pl['lead_days'].min(),
      ', average', round(pl['lead_days'].mean(), 2),
      ', largest', pl['lead_days'].max(), '(expected 7, about 17.9, 35)')

# waiting time per product
by_product = pl.groupby('product_id')['lead_days'].agg(['count', 'mean', 'std', 'min', 'max'])
print()
print(by_product.round(2))

Received purchase lines 140077 (expected 140077)
waiting days over all lines: smallest 7 , average 17.9 , largest 35 (expected 7, about 17.9, 35)

            count   mean   std  min  max
product_id                              
P001         4635  17.71  6.62    7   35
P002         4752  17.96  6.72    7   35
P003         4739  17.78  6.71    7   35
P004         4613  17.95  6.64    7   35
P005         4774  18.11  6.62    7   35
P006         4685  17.93  6.63    7   35
P007         4718  17.82  6.74    7   35
P008         4621  17.63  6.60    7   35
P009         4614  17.69  6.62    7   35
P010         4751  18.03  6.72    7   35
P011         4730  18.01  6.65    7   35
P012         4753  17.95  6.73    7   35
P013         4655  17.81  6.61    7   35
P014         4796  17.93  6.65    7   35
P015         4605  18.04  6.73    7   35
P016         4581  17.85  6.71    7   35
P017         4726  17.93  6.65    7   35
P018         4730  17.87  6.74    7   35
P019         4539  17.87  6.67   

In [4]:
# Delivered sales lines with their order date and branch
delivered = so_h[so_h['order_status'] == 'Delivered']
sales = so_l.merge(delivered[['so_id', 'order_date', 'branch_id']], on='so_id', how='inner')
sales['order_date'] = pd.to_datetime(sales['order_date'])
print('Delivered lines', len(sales), '(expected 117656)')
print('Delivered units', sales['quantity'].sum(), '(expected 1234939)')

# units sold per day for each product and branch pair, with zero for days without sales
all_days = pd.date_range('2019-01-01', '2024-12-31', freq='D')
daily = (sales.groupby(['product_id', 'branch_id', 'order_date'])['quantity'].sum()
         .unstack('order_date', fill_value=0)
         .reindex(columns=all_days, fill_value=0))
print('days', len(all_days), '(expected 2192)')
print('pairs by days', daily.shape, '(expected (180, 2192))')
print('units in the daily table', daily.values.sum(), '(expected 1234939)')

# average, spread and share of zero days for each pair
pair_stats = pd.DataFrame({
    'avg_daily': daily.mean(axis=1),
    'std_daily': daily.std(axis=1),
    'zero_day_share': (daily == 0).mean(axis=1)})
print()
print(pair_stats.describe().round(3))

Delivered lines 117656 (expected 117656)
Delivered units 1234939 (expected 1234939)
days 2192 (expected 2192)
pairs by days (180, 2192) (expected (180, 2192))
units in the daily table 1234939 (expected 1234939)

       avg_daily  std_daily  zero_day_share
count    180.000    180.000         180.000
mean       3.130      7.088           0.767
std        0.478      0.572           0.030
min        2.318      5.858           0.707
25%        2.728      6.595           0.743
50%        3.062      6.976           0.768
75%        3.527      7.585           0.794
max        4.042      8.263           0.820


In [5]:
import numpy as np

# each pair with its product's own average wait and spread of the wait
rp = pair_stats.reset_index()
waits = by_product[['mean', 'std']].rename(columns={'mean': 'wait_mean', 'std': 'wait_std'}).reset_index()
rp = rp.merge(waits, on='product_id', how='left')

# demand during the wait, safety stock for 95 percent of cases, and the reorder point
z = 1.645
rp['demand_in_wait'] = rp['avg_daily'] * rp['wait_mean']
rp['safety_stock_calc'] = z * np.sqrt(rp['wait_mean'] * rp['std_daily'] ** 2 + rp['avg_daily'] ** 2 * rp['wait_std'] ** 2)
rp['reorder_point'] = rp['demand_in_wait'] + rp['safety_stock_calc']

print('pairs', len(rp), '(expected 180)')
print('empty values', rp.isnull().sum().sum())
print()
print(rp[['demand_in_wait', 'safety_stock_calc', 'reorder_point']].describe().round(1))
print()
print(rp.head(3).round(2))

# stored values in the inventory table, for comparison only
print()
print(inv[['reorder_level', 'safety_stock', 'max_stock']].describe().round(1))

pairs 180 (expected 180)
empty values 0

       demand_in_wait  safety_stock_calc  reorder_point
count           180.0              180.0          180.0
mean             56.0               60.1          116.2
std               8.6                6.2           14.7
min              41.8               48.1           89.9
25%              48.7               54.7          103.5
50%              54.9               58.8          112.9
75%              63.2               65.2          128.1
max              73.4               72.7          144.5

  product_id branch_id  avg_daily  std_daily  zero_day_share  wait_mean  \
0       P001    AHM001       3.20       7.26            0.76      17.71   
1       P001    CHN001       3.96       8.17            0.72      17.71   
2       P001    DEL001       2.94       6.95            0.78      17.71   

   wait_std  demand_in_wait  safety_stock_calc  reorder_point  
0      6.62           56.60              61.13         117.72  
1      6.62           70.

In [6]:
# change in balance on every ledger row, starting each pair from its opening stock
ledger_c = ledger.merge(inv[['product_id', 'branch_id', 'opening_stock']], on=['product_id', 'branch_id'], how='left')
ledger_c['prev_balance'] = ledger_c.groupby(['product_id', 'branch_id'])['running_balance'].shift(1)
ledger_c['prev_balance'] = ledger_c['prev_balance'].fillna(ledger_c['opening_stock'])
ledger_c['change'] = ledger_c['running_balance'] - ledger_c['prev_balance']

# net adjustments, Received units and Delivered units for each pair
net_adj = ledger_c[ledger_c['movement_type'] == 'ADJUSTMENT'].groupby(['product_id', 'branch_id'])['change'].sum()
received_units = pl.groupby(['product_id', 'branch_id'])['quantity'].sum()
delivered_units = sales.groupby(['product_id', 'branch_id'])['quantity'].sum()

# order-line stock for each pair
stock = inv.set_index(['product_id', 'branch_id'])[['opening_stock']].copy()
stock['received'] = received_units
stock['delivered'] = delivered_units
stock['net_adj'] = net_adj.reindex(stock.index).fillna(0)
stock['order_line_stock'] = stock['opening_stock'] + stock['received'] - stock['delivered'] + stock['net_adj']

print('pairs', len(stock), '(expected 180)')
print('empty values', stock.isnull().sum().sum())
print('Received units', stock['received'].sum(), '(expected 22387123)')
print('Delivered units', stock['delivered'].sum(), '(expected 1234939)')
print('net adjustments', stock['net_adj'].sum(), '(expected 62)')
print('order-line stock total', stock['order_line_stock'].sum(), '(expected 21185215)')
print('smallest pair', stock['order_line_stock'].min(), '(expected 98642)')
print('largest pair', stock['order_line_stock'].max(), '(expected 132967)')

pairs 180 (expected 180)
empty values 0
Received units 22387123 (expected 22387123)
Delivered units 1234939 (expected 1234939)
net adjustments 62.0 (expected 62)
order-line stock total 21185215.0 (expected 21185215)
smallest pair 98642.0 (expected 98642)
largest pair 132967.0 (expected 132967)


In [7]:
# put stock, our reorder point and the stored values side by side for each pair
t = rp.set_index(['product_id', 'branch_id']).join(stock[['order_line_stock']])
stored = inv.set_index(['product_id', 'branch_id'])[['reorder_level', 'safety_stock']]
stored = stored.rename(columns={'reorder_level': 'stored_reorder_level', 'safety_stock': 'stored_safety_stock'})
t = t.join(stored)
t['times_reorder_point'] = t['order_line_stock'] / t['reorder_point']
t['days_of_demand'] = t['order_line_stock'] / t['avg_daily']

print('pairs', len(t), '(expected 180)')
print('empty values', t[['order_line_stock', 'reorder_point', 'stored_reorder_level']].isnull().sum().sum())
print()
print('pairs with stock at or below our reorder point', (t['order_line_stock'] <= t['reorder_point']).sum())
print('pairs with stock at or below the stored reorder level', (t['order_line_stock'] <= t['stored_reorder_level']).sum())
print('pairs where the stored reorder level is below our reorder point', (t['stored_reorder_level'] < t['reorder_point']).sum())
print('pairs where the stored reorder level is at or above our reorder point', (t['stored_reorder_level'] >= t['reorder_point']).sum())
print()
print(t[['order_line_stock', 'times_reorder_point', 'days_of_demand']].describe().round(1))
print()
print(t.sort_values('times_reorder_point').head(3)[['order_line_stock', 'reorder_point', 'times_reorder_point', 'days_of_demand']].round(1))

pairs 180 (expected 180)
empty values 0

pairs with stock at or below our reorder point 0
pairs with stock at or below the stored reorder level 0
pairs where the stored reorder level is below our reorder point 175
pairs where the stored reorder level is at or above our reorder point 5

       order_line_stock  times_reorder_point  days_of_demand
count             180.0                180.0           180.0
mean           117695.6               1030.5         38529.6
std              5367.5                146.7          6422.0
min             98642.0                745.7         27356.2
25%            113562.2                907.0         33159.6
50%            117900.5               1020.0         38473.1
75%            121422.5               1148.9         43772.3
max            132967.0               1377.2         53845.1

                      order_line_stock  reorder_point  times_reorder_point  \
product_id branch_id                                                         
P019   

In [8]:
# years until each pair would reach its reorder point if no new stock arrived
t['years_until_reorder_point'] = (t['order_line_stock'] - t['reorder_point']) / t['avg_daily'] / 365

# one row per branch
by_branch = t.reset_index().groupby('branch_id').agg(
    pairs=('product_id', 'size'),
    stock=('order_line_stock', 'sum'),
    reorder_points=('reorder_point', 'sum'),
    shortest_years=('years_until_reorder_point', 'min'),
    median_years=('years_until_reorder_point', 'median'))
print(by_branch.round(1))
print('pairs added up', by_branch['pairs'].sum(), '(expected 180)')
print('stock added up', by_branch['stock'].sum(), '(expected 21185215)')

# the 5 products closest to their reorder point (shortest time over their branches)
by_product = t.reset_index().groupby('product_id')['years_until_reorder_point'].min().sort_values().head(5)
print()
print(by_product.round(1))

           pairs      stock  reorder_points  shortest_years  median_years
branch_id                                                                
AHM001        30  3472113.0          3609.1            88.0          97.4
CHN001        30  3548773.0          4154.1            75.8          84.4
DEL001        30  3597678.0          3199.8           102.1         116.6
HYD001        30  3555196.0          2967.4           109.2         124.8
KOL001        30  3435814.0          3836.1            74.8          90.8
PUN001        30  3575641.0          3145.0           107.0         118.2
pairs added up 180 (expected 180)
stock added up 21185215.0 (expected 21185215)

product_id
P019    74.8
P027    75.8
P001    76.3
P030    77.8
P023    78.7
Name: years_until_reorder_point, dtype: float64


In [9]:
# blanks in the columns used, table by table
received_check = po_h[po_h['po_status'] == 'Received']
used = [('Received purchase headers', received_check, ['po_id', 'branch_id', 'order_date', 'received_date']),
        ('purchase lines', po_l, ['po_id', 'product_id', 'quantity']),
        ('Delivered sales headers', delivered, ['so_id', 'branch_id', 'order_date']),
        ('sales lines', so_l, ['so_id', 'product_id', 'quantity']),
        ('inventory table', inv, ['product_id', 'branch_id', 'opening_stock']),
        ('stock ledger', ledger, ['product_id', 'branch_id', 'movement_type', 'running_balance'])]
for name, table, cols in used:
    print(name, '| rows', len(table), '| blanks per column:', table[cols].isnull().sum().to_dict())

# zeros and negatives in the number columns used
print()
for name, series in [('purchase line quantity', po_l['quantity']), ('sales line quantity', so_l['quantity']),
                     ('opening stock', inv['opening_stock']), ('ledger running balance', ledger['running_balance'])]:
    print(name, '| zeros', (series == 0).sum(), '| negatives', (series < 0).sum(),
          '| smallest', series.min(), '| largest', series.max())

# waiting time per product, rebuilt under a new name
wait_table = pl.groupby('product_id')['lead_days'].agg(['mean', 'std'])

# blanks, zeros and negatives in our own tables
print()
own = [('wait per product', wait_table, ['mean', 'std']),
       ('daily demand per pair', pair_stats, ['avg_daily', 'std_daily']),
       ('reorder point table', rp, ['demand_in_wait', 'safety_stock_calc', 'reorder_point']),
       ('stock table', stock, ['order_line_stock'])]
for name, table, cols in own:
    print(name, '| rows', len(table), '| blanks', table[cols].isnull().sum().sum(),
          '| zeros or negatives', (table[cols] <= 0).sum().sum())

Received purchase headers | rows 21630 | blanks per column: {'po_id': 0, 'branch_id': 0, 'order_date': 0, 'received_date': 0}
purchase lines | rows 155495 | blanks per column: {'po_id': 0, 'product_id': 0, 'quantity': 0}
Delivered sales headers | rows 18033 | blanks per column: {'so_id': 0, 'branch_id': 0, 'order_date': 0}
sales lines | rows 130402 | blanks per column: {'so_id': 0, 'product_id': 0, 'quantity': 0}
inventory table | rows 180 | blanks per column: {'product_id': 0, 'branch_id': 0, 'opening_stock': 0}
stock ledger | rows 237230 | blanks per column: {'product_id': 0, 'branch_id': 0, 'movement_type': 0, 'running_balance': 0}

purchase line quantity | zeros 0 | negatives 0 | smallest 20 | largest 300
sales line quantity | zeros 0 | negatives 0 | smallest 1 | largest 20
opening stock | zeros 0 | negatives 0 | smallest 80 | largest 300
ledger running balance | zeros 0 | negatives 0 | smallest 14 | largest 121013

wait per product | rows 30 | blanks 0 | zeros or negatives 0
daily

In [10]:
# the pairs found in each table, compared with the inventory table
pair_sets = {'inventory table': set(zip(inv['product_id'], inv['branch_id'])),
             'stock ledger': set(zip(ledger['product_id'], ledger['branch_id'])),
             'Delivered sales': set(zip(sales['product_id'], sales['branch_id'])),
             'Received purchases': set(zip(pl['product_id'], pl['branch_id'])),
             'reorder point table': set(zip(rp['product_id'], rp['branch_id']))}
base = pair_sets['inventory table']
for name, pairs in pair_sets.items():
    print(name, '| pairs', len(pairs), '| same as inventory table', pairs == base)

# how much activity each pair has
days_with_sale = (daily > 0).sum(axis=1)
purchase_lines_per_pair = pl.groupby(['product_id', 'branch_id']).size()
print()
print('days with a sale per pair: smallest', days_with_sale.min(), ', largest', days_with_sale.max(), '(out of 2192 days)')
print('Received purchase lines per pair: smallest', purchase_lines_per_pair.min(), ', largest', purchase_lines_per_pair.max())

# lines left out because their order was not Received or not Delivered
print()
print('purchase lines left out', len(po_l) - len(pl), '(expected 15418)')
print('sales lines left out', len(so_l) - len(sales), '(expected 12746)')
cancelled_po = po_h.loc[po_h['po_status'] == 'Cancelled', 'po_id']
cancelled_so = so_h.loc[so_h['order_status'] == 'Cancelled', 'so_id']
print('lines of Cancelled purchase orders inside the purchase table used', pl['po_id'].isin(cancelled_po).sum())
print('lines of Cancelled sales orders inside the sales table used', sales['so_id'].isin(cancelled_so).sum())

# order statuses
print()
print(po_h['po_status'].value_counts().to_dict())
print(so_h['order_status'].value_counts().to_dict())

inventory table | pairs 180 | same as inventory table True
stock ledger | pairs 180 | same as inventory table True
Delivered sales | pairs 180 | same as inventory table True
Received purchases | pairs 180 | same as inventory table True
reorder point table | pairs 180 | same as inventory table True

days with a sale per pair: smallest 395 , largest 643 (out of 2192 days)
Received purchase lines per pair: smallest 671 , largest 842

purchase lines left out 15418 (expected 15418)
sales lines left out 12746 (expected 12746)
lines of Cancelled purchase orders inside the purchase table used 0
lines of Cancelled sales orders inside the sales table used 0

{'Received': 21630, 'Cancelled': 2370}
{'Delivered': 18033, 'Cancelled': 1967}


In [11]:
from scipy.stats import binomtest

# order ids used once, and no repeated order id with line number
print('purchase order ids in header', len(po_h), '| distinct', po_h['po_id'].nunique())
print('sales order ids in header', len(so_h), '| distinct', so_h['so_id'].nunique())
print('purchase lines with a repeated order id and line number', po_l.duplicated(['po_id', 'line_number']).sum())
print('sales lines with a repeated order id and line number', so_l.duplicated(['so_id', 'line_number']).sum())

# lines identical to another line apart from the line number, and the same product twice in one order
def repeated_lines(lines, order_col, original_columns, name):
    same_columns = [c for c in original_columns if c != 'line_number']
    print()
    print(name, '| lines', len(lines))
    print('lines inside a group of identical lines', lines.duplicated(same_columns, keep=False).sum())
    print('extra copies if one of each group is kept', lines.duplicated(same_columns, keep='first').sum())
    groups = lines.groupby([order_col, 'product_id'])['quantity'].agg(['size', 'min', 'max'])
    repeated = groups[groups['size'] > 1]
    print('order and product combinations with more than one line', len(repeated))
    print('combinations by number of lines', repeated['size'].value_counts().sort_index().to_dict())
    two = groups[groups['size'] == 2]
    equal = int((two['min'] == two['max']).sum())
    share = lines['quantity'].value_counts(normalize=True)
    chance = float((share ** 2).sum())
    result = binomtest(equal, len(two), chance)
    print('two-line combinations', len(two), '| equal quantity', equal, '| share', round(100 * equal / len(two), 2),
          '% | chance', round(100 * chance, 2), '% | p-value', round(result.pvalue, 4))

repeated_lines(sales, 'so_id', list(so_l.columns), 'Delivered sales lines')
repeated_lines(pl, 'po_id', list(po_l.columns), 'Received purchase lines')

purchase order ids in header 24000 | distinct 24000
sales order ids in header 20000 | distinct 20000
purchase lines with a repeated order id and line number 0
sales lines with a repeated order id and line number 0

Delivered sales lines | lines 117656
lines inside a group of identical lines 1374
extra copies if one of each group is kept 688
order and product combinations with more than one line 12184
combinations by number of lines {2: 11202, 3: 914, 4: 65, 5: 2, 6: 1}
two-line combinations 11202 | equal quantity 547 | share 4.88 % | chance 5.0 % | p-value 0.5879

Received purchase lines | lines 140077
lines inside a group of identical lines 0
extra copies if one of each group is kept 0
order and product combinations with more than one line 14408
combinations by number of lines {2: 13219, 3: 1129, 4: 57, 5: 3}
two-line combinations 13219 | equal quantity 53 | share 0.4 % | chance 0.36 % | p-value 0.3806


In [12]:
# used lines plus left-out lines against all lines
left_po = po_l[~po_l['po_id'].isin(received_check['po_id'])]
left_so = so_l[~so_l['so_id'].isin(delivered['so_id'])]
print('Received purchase lines', len(pl), '| left out', len(left_po), '| together', len(pl) + len(left_po), '| all purchase lines', len(po_l))
print('Delivered sales lines', len(sales), '| left out', len(left_so), '| together', len(sales) + len(left_so), '| all sales lines', len(so_l))
print('Received units', pl['quantity'].sum(), '(expected 22387123) | Delivered units', sales['quantity'].sum(), '(expected 1234939)')

# daily demand table against the sales lines
sales_by_pair = sales.groupby(['product_id', 'branch_id'])['quantity'].sum()
print()
print('units in the daily demand table', daily.values.sum(), '(expected 1234939)')
print('largest difference per pair, daily table against sales lines', (daily.sum(axis=1) - sales_by_pair).abs().max())
print('average daily demand times 2192 days, added up', round((pair_stats['avg_daily'] * len(all_days)).sum(), 1), '(expected 1234939)')

# stock built from its parts
print()
print('opening', stock['opening_stock'].sum(), '| Received', stock['received'].sum(), '| Delivered', stock['delivered'].sum(), '| net adjustments', stock['net_adj'].sum())
print('opening + Received - Delivered + net adjustments',
      stock['opening_stock'].sum() + stock['received'].sum() - stock['delivered'].sum() + stock['net_adj'].sum(), '(expected 21185215)')
print('stock in the final table', t['order_line_stock'].sum(), '(expected 21185215)')

# tie to the ledger
ledger_last = ledger.groupby(['product_id', 'branch_id'])['running_balance'].last()
print('ledger last balances added up', ledger_last.sum(), '(expected 19305994)')
print('order-line stock minus ledger', stock['order_line_stock'].sum() - ledger_last.sum(), '(expected 1879221)')

# reorder point built from its parts
print()
print('largest difference, reorder point minus demand during the wait minus safety stock',
      (rp['reorder_point'] - rp['demand_in_wait'] - rp['safety_stock_calc']).abs().max())

Received purchase lines 140077 | left out 15418 | together 155495 | all purchase lines 155495
Delivered sales lines 117656 | left out 12746 | together 130402 | all sales lines 130402
Received units 22387123 (expected 22387123) | Delivered units 1234939 (expected 1234939)

units in the daily demand table 1234939 (expected 1234939)
largest difference per pair, daily table against sales lines 0
average daily demand times 2192 days, added up 1234939.0 (expected 1234939)

opening 32969 | Received 22387123 | Delivered 1234939 | net adjustments 62.0
opening + Received - Delivered + net adjustments 21185215.0 (expected 21185215)
stock in the final table 21185215.0 (expected 21185215)
ledger last balances added up 19305994 (expected 19305994)
order-line stock minus ledger 1879221.0 (expected 1879221)

largest difference, reorder point minus demand during the wait minus safety stock 1.4210854715202004e-14


In [13]:
import math
from collections import defaultdict

# demand per pair per day, built by looping over the sales lines
n_days = len(all_days)
day_units = defaultdict(lambda: defaultdict(int))
for p, b, d, q in zip(sales['product_id'], sales['branch_id'], sales['order_date'], sales['quantity']):
    day_units[(p, b)][d] += q

# waiting time per product, built by looping over the purchase lines
wait_sum, wait_sq, wait_n = defaultdict(float), defaultdict(float), defaultdict(int)
for p, w in zip(pl['product_id'], pl['lead_days']):
    wait_sum[p] += w
    wait_sq[p] += w * w
    wait_n[p] += 1

# units received and delivered per pair, built by looping
received_loop, delivered_loop = defaultdict(int), defaultdict(int)
for p, b, q in zip(pl['product_id'], pl['branch_id'], pl['quantity']):
    received_loop[(p, b)] += q
for p, b, q in zip(sales['product_id'], sales['branch_id'], sales['quantity']):
    delivered_loop[(p, b)] += q

# net adjustments per pair, from the change in balance on each adjustment row in file order
opening_loop = {(p, b): o for p, b, o in zip(inv['product_id'], inv['branch_id'], inv['opening_stock'])}
last_balance, adjust_loop = {}, defaultdict(int)
for p, b, m, r in zip(ledger['product_id'], ledger['branch_id'], ledger['movement_type'], ledger['running_balance']):
    previous = last_balance.get((p, b), opening_loop[(p, b)])
    if m == 'ADJUSTMENT':
        adjust_loop[(p, b)] += r - previous
    last_balance[(p, b)] = r

# one row per pair with every number worked out by hand-written formulas
rows = []
for (p, b), per_day in day_units.items():
    total = sum(per_day.values())
    squares = sum(v * v for v in per_day.values())
    avg = total / n_days
    var_day = (squares - n_days * avg * avg) / (n_days - 1)
    n = wait_n[p]
    wmean = wait_sum[p] / n
    wvar = (wait_sq[p] - n * wmean * wmean) / (n - 1)
    safety = 1.645 * math.sqrt(wmean * var_day + avg * avg * wvar)
    stock_loop = opening_loop[(p, b)] + received_loop[(p, b)] - delivered_loop[(p, b)] + adjust_loop[(p, b)]
    rows.append((p, b, avg, math.sqrt(var_day), wmean, math.sqrt(wvar), wmean * avg, safety, wmean * avg + safety, stock_loop))
loop = pd.DataFrame(rows, columns=['product_id', 'branch_id', 'avg_daily', 'std_daily', 'wait_mean', 'wait_std',
                                   'demand_in_wait', 'safety_stock_calc', 'reorder_point', 'order_line_stock'])
loop = loop.set_index(['product_id', 'branch_id'])

# largest difference between the two methods
tables_side = rp.set_index(['product_id', 'branch_id']).join(stock[['order_line_stock']])
print('pairs in the loop method', len(loop), '(expected 180)')
for col in loop.columns:
    print(col, '| largest difference', (loop[col] - tables_side[col]).abs().max())
print('pairs at or below their reorder point (loop method)', (loop['order_line_stock'] <= loop['reorder_point']).sum())
print('smallest stock over reorder point (loop method)', round((loop['order_line_stock'] / loop['reorder_point']).min(), 1), '(expected 745.7)')

pairs in the loop method 180 (expected 180)
avg_daily | largest difference 0.0
std_daily | largest difference 2.1316282072803006e-13
wait_mean | largest difference 0.0
wait_std | largest difference 1.509903313490213e-14
demand_in_wait | largest difference 0.0
safety_stock_calc | largest difference 1.1937117960769683e-12
reorder_point | largest difference 1.1937117960769683e-12
order_line_stock | largest difference 0.0
pairs at or below their reorder point (loop method) 0
smallest stock over reorder point (loop method) 745.7 (expected 745.7)


In [14]:
# reorder points at three service levels
rows = []
for name, z in [('90%', 1.282), ('95%', 1.645), ('99%', 2.326)]:
    safety = z * np.sqrt(rp['wait_mean'] * rp['std_daily'] ** 2 + rp['avg_daily'] ** 2 * rp['wait_std'] ** 2)
    level = rp['demand_in_wait'] + safety
    stock_values = t['order_line_stock'].values
    rows.append((name, z, level.mean(), level.max(), int((stock_values <= level.values).sum()), (stock_values / level.values).min()))
sens = pd.DataFrame(rows, columns=['service level', 'z', 'average reorder point', 'largest reorder point',
                                   'pairs at or below', 'smallest stock over reorder point'])
print(sens.round(1))

# worst case: every order takes the longest wait seen, 35 days
worst = rp['avg_daily'] * 35 + 1.645 * np.sqrt(35 * rp['std_daily'] ** 2)
print()
print('longest wait', pl['lead_days'].max(), 'over average wait', round(pl['lead_days'].max() / pl['lead_days'].mean(), 2))
print('largest reorder point if every wait were 35 days', round(worst.max(), 1),
      '| pairs at or below', int((t['order_line_stock'].values <= worst.values).sum()),
      '| smallest stock over it', round((t['order_line_stock'].values / worst.values).min(), 1))

# the most units each pair really sold in any 35 days, against its stock
largest_35 = daily.T.rolling(35).sum().max()
stock_aligned = t['order_line_stock'].reindex(largest_35.index)
print()
print('most units any pair sold in 35 days', largest_35.max(), '| average pair', round(largest_35.mean(), 1))
print('pairs with stock below their own largest 35-day demand', int((stock_aligned < largest_35).sum()))
print('smallest stock over largest 35-day demand', round((stock_aligned / largest_35).min(), 1))

# single-day outliers
biggest_day = daily.max(axis=1)
ratio_day = biggest_day / pair_stats['avg_daily']
print()
print('largest single-day sale of any pair', biggest_day.max(), '| average pair biggest day', round(biggest_day.mean(), 1))
print('biggest day over the pair daily average: smallest', round(ratio_day.min(), 1), '| largest', round(ratio_day.max(), 1))

  service level    z  average reorder point  largest reorder point  \
0           90%  1.3                  102.9                  128.8   
1           95%  1.6                  116.2                  144.5   
2           99%  2.3                  141.1                  174.6   

   pairs at or below  smallest stock over reorder point  
0                  0                              840.9  
1                  0                              745.7  
2                  0                              615.1  

longest wait 35 over average wait 1.96
largest reorder point if every wait were 35 days 221.1 | pairs at or below 0 | smallest stock over it 485.1

most units any pair sold in 35 days 383.0 | average pair 256.0
pairs with stock below their own largest 35-day demand 0
smallest stock over largest 35-day demand 285.5

largest single-day sale of any pair 92 | average pair biggest day 57.4
biggest day over the pair daily average: smallest 11.8 | largest 30.5


In [15]:
# date ranges of the inputs
print('Delivered sales: first order', sales['order_date'].min().date(), '| last order', sales['order_date'].max().date())
delivery_dates = pd.to_datetime(delivered['delivery_date'])
print('Delivered sales: first delivery', delivery_dates.min().date(), '| last delivery', delivery_dates.max().date())
print('Received purchases: first order', pd.to_datetime(received['order_date']).min().date(),
      '| last order', pd.to_datetime(received['order_date']).max().date())
print('Received purchases: first receipt', pd.to_datetime(received['received_date']).min().date(),
      '| last receipt', pd.to_datetime(received['received_date']).max().date())
ledger_dates = pd.to_datetime(ledger['movement_date'])
print('stock ledger: first entry', ledger_dates.min().date(), '| last entry', ledger_dates.max().date())
print('Received orders with a wait of zero days or less', int((received['lead_days'] <= 0).sum()))

# waits and demand by year
pl_dates = pl.merge(received[['po_id', 'order_date']], on='po_id', how='left')
pl_dates['year'] = pd.to_datetime(pl_dates['order_date']).dt.year
sales['year'] = sales['order_date'].dt.year
days_in_year = pd.Series({2019: 365, 2020: 366, 2021: 365, 2022: 365, 2023: 365, 2024: 366})
by_year = pd.DataFrame({'Received lines': pl_dates.groupby('year').size(),
                        'average wait days': pl_dates.groupby('year')['lead_days'].mean().round(2),
                        'units sold': sales.groupby('year')['quantity'].sum()})
by_year['average daily demand per pair'] = (by_year['units sold'] / days_in_year / 180).round(3)
print()
print(by_year)
print('Received lines added up', by_year['Received lines'].sum(), '(expected 140077) | units sold added up', by_year['units sold'].sum(), '(expected 1234939)')

# months covered
pl_dates['month'] = pd.to_datetime(pl_dates['order_date']).dt.to_period('M')
months_per_product = pl_dates.groupby('product_id')['month'].nunique()
print()
print('months with Received lines per product: smallest', months_per_product.min(), '| largest', months_per_product.max(), '(expected 72)')
sales['month'] = sales['order_date'].dt.to_period('M')
months_per_pair = sales.groupby(['product_id', 'branch_id'])['month'].nunique()
print('months with sales per pair: smallest', months_per_pair.min(), '| largest', months_per_pair.max(), '(expected smallest 70, largest 72)')

Delivered sales: first order 2019-01-01 | last order 2024-12-31
Delivered sales: first delivery 2019-01-03 | last delivery 2025-01-13
Received purchases: first order 2019-01-01 | last order 2024-12-31
Received purchases: first receipt 2019-01-11 | last receipt 2025-01-28
stock ledger: first entry 2019-01-01 | last entry 2025-01-28
Received orders with a wait of zero days or less 0

      Received lines  average wait days  units sold  \
year                                                  
2019           23431              18.12      207925   
2020           23773              17.96      212007   
2021           23443              17.91      208269   
2022           23097              17.78      202242   
2023           23377              17.79      204638   
2024           22956              17.84      199858   

      average daily demand per pair  
year                                 
2019                          3.165  
2020                          3.218  
2021                  

In [16]:
import csv
import math
import datetime

def read_rows(name):
    with open(BASE_PATH + name, newline='') as f:
        return list(csv.DictReader(f))

def to_date(text):
    return datetime.date.fromisoformat(text[:10])

# the raw files, read with plain Python
so_header = {r['so_id']: r for r in read_rows('sales_orders_header.csv')}
po_header = {r['po_id']: r for r in read_rows('purchase_orders_header.csv')}
so_lines = read_rows('sales_orders_lines.csv')
po_lines = read_rows('purchase_orders_lines.csv')
inv_rows = read_rows('inventory_master.csv')
ledger_rows = read_rows('stock_ledger.csv')
days = (datetime.date(2024, 12, 31) - datetime.date(2019, 1, 1)).days + 1

for product, branch in [('P019', 'KOL001'), ('P001', 'CHN001'), ('P004', 'HYD001')]:
    # Delivered sales of this pair, units per day
    per_day = {}
    for r in so_lines:
        if r['product_id'] == product:
            h = so_header[r['so_id']]
            if h['order_status'] == 'Delivered' and h['branch_id'] == branch:
                per_day[h['order_date']] = per_day.get(h['order_date'], 0) + int(r['quantity'])
    units_sold = sum(per_day.values())
    squares = sum(v * v for v in per_day.values())
    avg = units_sold / days
    var_day = (squares - days * avg * avg) / (days - 1)

    # waiting time of this product on Received orders, and units received in this branch
    n, wait_total, wait_squares, units_received = 0, 0.0, 0.0, 0
    for r in po_lines:
        if r['product_id'] == product:
            h = po_header[r['po_id']]
            if h['po_status'] == 'Received':
                wait = (to_date(h['received_date']) - to_date(h['order_date'])).days
                n += 1
                wait_total += wait
                wait_squares += wait * wait
                if h['branch_id'] == branch:
                    units_received += int(r['quantity'])
    wait_mean = wait_total / n
    wait_var = (wait_squares - n * wait_mean * wait_mean) / (n - 1)

    # opening stock and net adjustments from the ledger in file order
    opening = next(int(r['opening_stock']) for r in inv_rows if r['product_id'] == product and r['branch_id'] == branch)
    previous, adjustments = opening, 0
    for r in ledger_rows:
        if r['product_id'] == product and r['branch_id'] == branch:
            balance = int(r['running_balance'])
            if r['movement_type'] == 'ADJUSTMENT':
                adjustments += balance - previous
            previous = balance

    stock_by_hand = opening + units_received - units_sold + adjustments
    safety = 1.645 * math.sqrt(wait_mean * var_day + avg * avg * wait_var)
    reorder_by_hand = wait_mean * avg + safety

    row = t.loc[(product, branch)]
    print(product, branch)
    print('  by hand | days with a sale', len(per_day), '| units sold', units_sold, '| average daily', round(avg, 4),
          '| daily std', round(math.sqrt(var_day), 4), '| wait', round(wait_mean, 3), '| wait std', round(math.sqrt(wait_var), 3))
    print('  by hand | opening', opening, '| received', units_received, '| adjustments', adjustments, '| stock', stock_by_hand,
          '| safety', round(safety, 2), '| reorder point', round(reorder_by_hand, 2), '| times', round(stock_by_hand / reorder_by_hand, 1))
    print('  table   | average daily', round(row['avg_daily'], 4), '| daily std', round(row['std_daily'], 4),
          '| wait', round(row['wait_mean'], 3), '| wait std', round(row['wait_std'], 3), '| stock', row['order_line_stock'],
          '| safety', round(row['safety_stock_calc'], 2), '| reorder point', round(row['reorder_point'], 2),
          '| times', round(row['times_reorder_point'], 1))
    print('  raw files and table match',
          abs(avg - row['avg_daily']) < 1e-9 and abs(math.sqrt(var_day) - row['std_daily']) < 1e-9
          and abs(wait_mean - row['wait_mean']) < 1e-9 and abs(math.sqrt(wait_var) - row['wait_std']) < 1e-9
          and stock_by_hand == row['order_line_stock'] and abs(reorder_by_hand - row['reorder_point']) < 1e-6)

P019 KOL001
  by hand | days with a sale 560 | units sold 7904 | average daily 3.6058 | daily std 7.9247 | wait 17.868 | wait std 6.673
  by hand | opening 92 | received 106282 | adjustments 172 | stock 98642 | safety 67.85 | reorder point 132.28 | times 745.7
  table   | average daily 3.6058 | daily std 7.9247 | wait 17.868 | wait std 6.673 | stock 98642.0 | safety 67.85 | reorder point 132.28 | times 745.7
  raw files and table match True
P001 CHN001
  by hand | days with a sale 614 | units sold 8681 | average daily 3.9603 | daily std 8.1704 | wait 17.707 | wait std 6.616
  by hand | opening 187 | received 119011 | adjustments -115 | stock 110402 | safety 71.11 | reorder point 141.24 | times 781.7
  table   | average daily 3.9603 | daily std 8.1704 | wait 17.707 | wait std 6.616 | stock 110402.0 | safety 71.11 | reorder point 141.24 | times 781.7
  raw files and table match True
P004 HYD001
  by hand | days with a sale 404 | units sold 5148 | average daily 2.3485 | daily std 6.0275 |

In [17]:
# stored lead time in the products table against the real average wait of each product
stored_days = products.set_index('product_id')['lead_time_days']
real_wait = pl.groupby('product_id')['lead_days'].mean()
compare = pd.DataFrame({'stored days': stored_days, 'real average wait': real_wait.round(2)})
compare['real minus stored'] = (compare['real average wait'] - compare['stored days']).round(2)

print('products compared', len(compare), '(expected 30)')
print('stored days: smallest', compare['stored days'].min(), '| largest', compare['stored days'].max(),
      '| average', round(compare['stored days'].mean(), 2))
print('real average wait: smallest', compare['real average wait'].min(), '| largest', compare['real average wait'].max(),
      '| average', round(compare['real average wait'].mean(), 2))
print('products where stored and real differ by more than 1 day', int((compare['real minus stored'].abs() > 1).sum()))
print('products where stored and real differ by more than 5 days', int((compare['real minus stored'].abs() > 5).sum()))
print()
print(compare.reindex(compare['real minus stored'].abs().sort_values(ascending=False).index).head(5))

products compared 30 (expected 30)
stored days: smallest 4 | largest 40 | average 17.0
real average wait: smallest 17.63 | largest 18.15 | average 17.9
products where stored and real differ by more than 1 day 27
products where stored and real differ by more than 5 days 19

            stored days  real average wait  real minus stored
product_id                                                   
P015                 40              18.04             -21.96
P014                 35              17.93             -17.07
P026                  4              17.93              13.93
P017                  5              17.93              12.93
P027                  5              17.85              12.85
